# Vision-Guided Pick-and-Place System
This notebook implements the full vision-guided pick-and-place system for the OpenManipulator-X, configured as requested in `template.txt`.
It integrates RealSense vision, ArUco detection, and Dynamixel SDK control.

In [73]:
import pyrealsense2 as rs
import numpy as np
import cv2
import math
import time
from dynamixel_sdk import *

In [74]:
# ============================================================
# 1. REALSENSE SETUP
# ============================================================
def init_realsense():
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)

    profile = pipeline.start(config)
    align_to = rs.stream.color
    align = rs.align(align_to)

    intrinsics = profile.get_stream(rs.stream.color).as_video_stream_profile().get_intrinsics()
    return pipeline, align, intrinsics

def get_frames(pipeline, align):
    frames = pipeline.wait_for_frames()
    aligned_frames = align.process(frames)
    color_frame = aligned_frames.get_color_frame()
    depth_frame = aligned_frames.get_depth_frame()

    if not color_frame or not depth_frame:
        return None, None
    color_image = np.asanyarray(color_frame.get_data())
    return color_image, depth_frame

In [75]:
# ============================================================
# 2. ARUCO DETECTION
# ============================================================
def init_aruco():
    try:
        aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
        detector_params = cv2.aruco.DetectorParameters()
        detector = cv2.aruco.ArucoDetector(aruco_dict, detector_params)
        return detector_params, aruco_dict, detector, False
    except AttributeError:
        aruco_dict = cv2.aruco.Dictionary_get(cv2.aruco.DICT_4X4_50)
        detector_params = cv2.aruco.DetectorParameters_create()
        return detector_params, aruco_dict, None, True

def detect_markers(image, aruco_dict, detector, OPENCV_OLD, detector_params):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if OPENCV_OLD:
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=detector_params)
    else:
        corners, ids, rejected = detector.detectMarkers(gray)
    return corners, ids

In [76]:
# ============================================================
# 3. WORKSPACE MAPPING (HOMOGRAPHY)
# ============================================================
def compute_homography(corners, ids, W=0.3, H=0.2):
    expected_ids = [0, 1, 2, 3]
    if ids is None: return None
    
    detected_ids = ids.flatten().tolist()
    if not all(req_id in detected_ids for req_id in expected_ids): return None

    image_pts = []
    world_pts = np.array([
        [0, 0], [W, 0], [W, H], [0, H]
    ], dtype=np.float32)

    for req_id in expected_ids:
        idx = detected_ids.index(req_id)
        marker_corners = corners[idx][0]
        center_x = np.mean(marker_corners[:, 0])
        center_y = np.mean(marker_corners[:, 1])
        image_pts.append([center_x, center_y])

    image_pts = np.array(image_pts, dtype=np.float32)
    H_matrix, _ = cv2.findHomography(image_pts, world_pts)
    return H_matrix

def pixel_to_workspace(u, v, H_matrix):
    pt = np.array([[[u, v]]], dtype=np.float32)
    pt_trans = cv2.perspectiveTransform(pt, H_matrix)
    return pt_trans[0][0][0], pt_trans[0][0][1]

In [77]:
# ============================================================
# 4. DEPTH & 5. ROBOT BASE CALIBRATION & 6. OBJECT POSITION
# ============================================================
def get_depth(depth_frame, u, v):
    return depth_frame.get_distance(u, v)

def get_base_transform(corners, ids, marker_size=0.05, camera_matrix=None, dist_coeffs=None):
    if ids is None: return None
    detected_ids = ids.flatten().tolist()
    if 4 not in detected_ids: return None

    idx = detected_ids.index(4)
    # Define the 3d coordinates of the 4 corners of the marker
    marker_points = np.array([
        [-marker_size / 2, marker_size / 2, 0],
        [marker_size / 2, marker_size / 2, 0],
        [marker_size / 2, -marker_size / 2, 0],
        [-marker_size / 2, -marker_size / 2, 0]
    ], dtype=np.float32)
    
    # Use solvePnP to get pose (replacing deprecated cv2.aruco.estimatePoseSingleMarkers)
    success, rvec, tvec = cv2.solvePnP(marker_points, corners[idx][0], camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_IPPE_SQUARE)
    if not success: return None
    
    rvec = rvec.flatten()
    tvec = tvec.flatten()
    
    R, _ = cv2.Rodrigues(rvec)
    T_camera_base = np.eye(4)
    T_camera_base[:3, :3] = R
    T_camera_base[:3, 3] = tvec
    
    # Invert
    return np.linalg.inv(T_camera_base)

def pixel_to_camera(u, v, Z, intrinsics):
    pt = rs.rs2_deproject_pixel_to_point(intrinsics, [u, v], Z)
    return np.array([pt[0], pt[1], pt[2], 1.0])

def transform_to_base(pt_camera, T_base_camera):
    P_base = T_base_camera @ pt_camera
    return P_base[:3]

In [78]:
# ============================================================
# 8. INVERSE KINEMATICS (OpenManipulator-X)
# ============================================================
ARM_LINKS = {
    "base_height": -0.160,
    "shoulder": 0.130,
    "elbow": 0.135,
    "wrist": 0.060,
}
JOINT_OFFSET_RAD = math.atan2(0.024, 0.128)

def inverse_kinematics(x, y, z):
    L1 = ARM_LINKS["base_height"]
    L2 = ARM_LINKS["shoulder"]
    L3 = ARM_LINKS["elbow"]
    L4 = ARM_LINKS["wrist"]

    joint1 = math.atan2(y, x)
    r = math.sqrt(x**2 + y**2)
    z_prime = z - L1

    # Keep the end-effector approximately level by solving IK for the wrist center.
    wrist_r = r - L4
    if wrist_r <= 0:
        return None

    reach = math.sqrt(wrist_r**2 + z_prime**2)
    min_reach = abs(L2 - L3) + 1e-6
    max_reach = (L2 + L3) - 1e-6
    if reach < min_reach or reach > max_reach:
        return None

    D = (wrist_r**2 + z_prime**2 - L2**2 - L3**2) / (2 * L2 * L3)
    D = max(-1.0, min(1.0, D))

    # Use the opposite branch so the manipulator approaches with wrist-down posture.
    elbow_eff = math.atan2(math.sqrt(max(0.0, 1 - D**2)), D)
    shoulder_eff = math.atan2(z_prime, wrist_r) - math.atan2(L3 * math.sin(elbow_eff), L2 + L3 * math.cos(elbow_eff))

    # The OpenManipulator links are mounted with a fixed mechanical offset.
    joint2 = shoulder_eff + JOINT_OFFSET_RAD
    joint3 = elbow_eff - JOINT_OFFSET_RAD
    joint4 = -(joint2 + joint3)
    return [joint1, joint2, joint3, joint4]


In [79]:
# ============================================================
# 9. DYNAMIXEL SDK SETUP & 10. POSITION & 11. SMOOTH MOTION
# ============================================================
ADDR_TORQUE_ENABLE = 64
ADDR_GOAL_POSITION = 116
ADDR_PRESENT_POSITION = 132
PROTOCOL_VERSION = 2.0
BAUDRATE = 1000000
DEVICENAME = 'COM9' # Match robot.ipynb
DXL_IDs = [11, 12, 13, 14, 15]
GRIPPER_OPEN_POS = 1500
GRIPPER_CLOSED_POS = 2300
DEFAULT_POS = {11: 2048, 12: 1227, 13: 2524, 14: 2414, 15: GRIPPER_OPEN_POS}

def init_dynamixel():
    portHandler = PortHandler(DEVICENAME)
    packetHandler = PacketHandler(PROTOCOL_VERSION)
    
    if not portHandler.openPort() or not portHandler.setBaudRate(BAUDRATE):
        return None, None
        
    for dxl_id in DXL_IDs:
        packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, 1)
    return portHandler, packetHandler

def rad_to_dynamixel(angle):
    pos = int((angle + math.pi) * (4095.0 / (2 * math.pi)))
    return max(0, min(4095, pos))

def smooth_profile(t):
    return 0.5 * (1 - math.cos(math.pi * t))

def send_joint_positions(portHandler, packetHandler, q_positions):
    for i, dxl_id in enumerate([11, 12, 13, 14]):
        pos = rad_to_dynamixel(q_positions[i])
        packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, pos)

def move_smooth(portHandler, packetHandler, q_start, q_goal, duration):
    steps = 80
    dt = duration / steps
    for step in range(steps + 1):
        t = step / steps
        alpha = smooth_profile(t)
        q = [qs + alpha * (qg - qs) for qs, qg in zip(q_start, q_goal)]
        send_joint_positions(portHandler, packetHandler, q)
        time.sleep(dt)

def set_gripper(portHandler, packetHandler, open_gripper=True):
    pos = GRIPPER_OPEN_POS if open_gripper else GRIPPER_CLOSED_POS
    packetHandler.write4ByteTxRx(portHandler, 15, ADDR_GOAL_POSITION, pos)

def read_positions(portHandler, packetHandler):
    positions = {}
    for dxl_id in DXL_IDs:
        pos, dxl_comm, dxl_err = packetHandler.read4ByteTxRx(portHandler, dxl_id, ADDR_PRESENT_POSITION)
        if dxl_comm == COMM_SUCCESS and dxl_err == 0:
            positions[dxl_id] = pos
    return positions

def set_default_positions(portHandler, packetHandler):
    print("Moving robot to default position using software interpolation...")
    
    current_positions = read_positions(portHandler, packetHandler)
    if not current_positions:
        print("Failed to read current positions. Moving directly to default...")
        for dxl_id, pos in DEFAULT_POS.items():
            packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, pos)
        time.sleep(1.5)
        return

    steps = 50
    step_delay = 0.03
    increments = {}
    
    for dxl_id in DEFAULT_POS:
        if dxl_id in current_positions:
            increments[dxl_id] = (DEFAULT_POS[dxl_id] - current_positions[dxl_id]) / steps
            
    for step in range(1, steps + 1):
        for dxl_id in DEFAULT_POS:
            if dxl_id in increments:
                int_pos = int(current_positions[dxl_id] + (increments[dxl_id] * step))
                int_pos = max(0, min(4095, int_pos))
                packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, int_pos)
        time.sleep(step_delay)


In [80]:
# ============================================================
# 14. MOTION SEQUENCE
# ============================================================
TARGET_FORWARD_OFFSET = 0.000
TARGET_LATERAL_OFFSET = 0.000
BOARD_PLANE_PICK_Z = 0.000
BOARD_PLANE_APPROACH_Z = 0.000
WORKSPACE_X_LIMITS = (0.10, 0.36)
WORKSPACE_Y_LIMIT = 0.20
ROBOT_BASE_FORWARD_OFFSET = 0.000
ROBOT_BASE_LATERAL_OFFSET = 0.000
ROBOT_BASE_VERTICAL_OFFSET = 0.000

def dynamixel_to_rad(pos):
    return (pos * (2 * math.pi) / 4095.0) - math.pi

def get_smoothed_depth(depth_frame, u, v, radius=2):
    values = []
    height = depth_frame.get_height()
    width = depth_frame.get_width()
    for dv in range(-radius, radius + 1):
        for du in range(-radius, radius + 1):
            uu = max(0, min(width - 1, u + du))
            vv = max(0, min(height - 1, v + dv))
            depth_value = depth_frame.get_distance(uu, vv)
            if depth_value > 0:
                values.append(depth_value)
    if not values:
        return 0.0
    return float(np.median(values))

def is_target_in_workspace(x, y):
    if x < WORKSPACE_X_LIMITS[0]:
        return False, f"x={x:.3f} is too close to the robot base"
    if x > WORKSPACE_X_LIMITS[1]:
        return False, f"x={x:.3f} is beyond the current workspace limit"
    if abs(y) > WORKSPACE_Y_LIMIT:
        return False, f"|y|={abs(y):.3f} is beyond the side-to-side limit"
    return True, "inside workspace"

def normalize_vector(vec):
    norm = np.linalg.norm(vec)
    if norm < 1e-9:
        return None
    return vec / norm

def project_point_to_plane(point, plane_point, plane_normal):
    return point - np.dot(point - plane_point, plane_normal) * plane_normal

def get_marker_center_camera_point(corners, ids, marker_id, depth_frame, intrinsics):
    if ids is None:
        return None
    detected_ids = ids.flatten().tolist()
    if marker_id not in detected_ids:
        return None

    idx = detected_ids.index(marker_id)
    marker_corners = corners[idx][0]
    u = int(np.mean(marker_corners[:, 0]))
    v = int(np.mean(marker_corners[:, 1]))
    depth_value = get_smoothed_depth(depth_frame, u, v)
    if depth_value <= 0:
        return None

    point = rs.rs2_deproject_pixel_to_point(intrinsics, [u, v], depth_value)
    return np.array(point, dtype=np.float32)

def build_robot_frame(corners, ids, depth_frame, intrinsics):
    required_ids = [0, 1, 2, 3, 4]
    marker_points = {}
    for marker_id in required_ids:
        point = get_marker_center_camera_point(corners, ids, marker_id, depth_frame, intrinsics)
        if point is None:
            return None, f"missing 3D point for marker {marker_id}"
        marker_points[marker_id] = point

    p0 = marker_points[0]
    p1 = marker_points[1]
    p2 = marker_points[2]
    p3 = marker_points[3]
    p4 = marker_points[4]

    board_right = normalize_vector(p1 - p0)
    board_forward_raw = p3 - p0
    board_forward_raw = board_forward_raw - np.dot(board_forward_raw, board_right) * board_right
    board_forward = normalize_vector(board_forward_raw)
    if board_right is None or board_forward is None:
        return None, "failed to build board axes"

    board_up = normalize_vector(np.cross(board_forward, board_right))
    if board_up is None:
        return None, "failed to build board normal"

    workspace_center = 0.25 * (p0 + p1 + p2 + p3)
    if np.dot(p4 - workspace_center, board_up) < 0:
        board_up = -board_up

    base_projection = project_point_to_plane(p4, p0, board_up)
    origin = (
        base_projection
        + ROBOT_BASE_FORWARD_OFFSET * board_forward
        + ROBOT_BASE_LATERAL_OFFSET * board_right
        + ROBOT_BASE_VERTICAL_OFFSET * board_up
    )

    return {
        "origin": origin,
        "x_axis": board_forward,
        "y_axis": board_right,
        "z_axis": board_up,
        "workspace_center": workspace_center,
        "base_marker_center": p4,
    }, None

def transform_camera_to_robot(point_camera, robot_frame):
    delta = point_camera[:3] - robot_frame["origin"]
    return np.array([
        float(np.dot(delta, robot_frame["x_axis"])),
        float(np.dot(delta, robot_frame["y_axis"])),
        float(np.dot(delta, robot_frame["z_axis"])),
    ])

def average_robot_frames(frames):
    if not frames:
        return None

    def avg_vec(key):
        return np.mean([frame[key] for frame in frames], axis=0)

    x_axis = normalize_vector(avg_vec("x_axis"))
    y_axis_seed = avg_vec("y_axis")
    y_axis_seed = y_axis_seed - np.dot(y_axis_seed, x_axis) * x_axis
    y_axis = normalize_vector(y_axis_seed)
    z_axis = normalize_vector(np.cross(x_axis, y_axis))
    y_axis = normalize_vector(np.cross(z_axis, x_axis))

    return {
        "origin": avg_vec("origin"),
        "x_axis": x_axis,
        "y_axis": y_axis,
        "z_axis": z_axis,
        "workspace_center": avg_vec("workspace_center"),
        "base_marker_center": avg_vec("base_marker_center"),
    }

def calibrate_workspace(pipeline, align, intrinsics, detector_params, aruco_dict, detector, OPENCV_OLD, stable_frames=8):
    print("Calibrating workspace. Keep markers 0,1,2,3,4 clearly visible...")
    collected_frames = []
    last_count = -1

    while True:
        color_image, depth_frame = get_frames(pipeline, align)
        if color_image is None:
            continue

        corners, ids = detect_markers(color_image, aruco_dict, detector, OPENCV_OLD, detector_params)
        display_image = color_image.copy()
        if ids is not None:
            cv2.aruco.drawDetectedMarkers(display_image, corners, ids)
        cv2.imshow("Camera Feed - ArUco Detection", display_image)
        cv2.waitKey(1)

        robot_frame, frame_error = build_robot_frame(corners, ids, depth_frame, intrinsics)
        if robot_frame is None:
            if last_count != 0:
                print(f"Calibration waiting: {frame_error}.")
                last_count = 0
            collected_frames.clear()
            continue

        collected_frames.append(robot_frame)
        if len(collected_frames) > stable_frames:
            collected_frames.pop(0)

        if len(collected_frames) != last_count:
            print(f"Calibration progress: {len(collected_frames)}/{stable_frames} stable frames")
            last_count = len(collected_frames)

        if len(collected_frames) >= stable_frames:
            calibrated_frame = average_robot_frames(collected_frames)
            print("Workspace calibration locked.")
            print(f"Origin: {[round(v, 3) for v in calibrated_frame['origin']]}")
            print(f"X axis: {[round(v, 3) for v in calibrated_frame['x_axis']]}")
            print(f"Y axis: {[round(v, 3) for v in calibrated_frame['y_axis']]}")
            print(f"Z axis: {[round(v, 3) for v in calibrated_frame['z_axis']]}")
            return calibrated_frame

def choose_target_marker(corners, ids):
    if ids is None:
        return None

    best = None
    detected_ids = ids.flatten().tolist()
    for i, marker_id in enumerate(detected_ids):
        if marker_id in [0, 1, 2, 3, 4]:
            continue

        marker_corners = corners[i][0]
        area = abs(cv2.contourArea(marker_corners.astype(np.float32)))
        center_u = int(np.mean(marker_corners[:, 0]))
        center_v = int(np.mean(marker_corners[:, 1]))
        candidate = (area, marker_id, center_u, center_v)
        if best is None or area > best[0]:
            best = candidate

    return best

def execute_task(pipeline, align, intrinsics, detector_params, aruco_dict, detector, OPENCV_OLD, portHandler, packetHandler, camera_matrix, dist_coeffs, robot_frame):
    """Executes the CV and target movement tasks. Returns boolean on success."""
    color_image, depth_frame = get_frames(pipeline, align)
    if color_image is None:
        return False
        
    corners, ids = detect_markers(color_image, aruco_dict, detector, OPENCV_OLD, detector_params)
    
    # --- VISUALIZATION ---
    display_image = color_image.copy()
    if ids is not None:
        cv2.aruco.drawDetectedMarkers(display_image, corners, ids)
    cv2.imshow("Camera Feed - ArUco Detection", display_image)
    cv2.waitKey(1)
    # ---------------------
    
    target = choose_target_marker(corners, ids)
    if target is None:
        return False

    target_area, target_id, u, v = target
    print(f"Target object detected (ID: {target_id}) at pixel ({u}, {v}), area={target_area:.1f}")

    Z = get_smoothed_depth(depth_frame, u, v)
    if Z <= 0:
        print("Depth lookup failed at target center. Retrying...")
        return False
        
    pt_cam = pixel_to_camera(u, v, Z, intrinsics)
    P_robot = transform_camera_to_robot(pt_cam[:3], robot_frame)
    tx, ty, raw_tz = P_robot.tolist()
    tx += TARGET_FORWARD_OFFSET
    ty += TARGET_LATERAL_OFFSET
    print(
        f"Robot-frame Target -> X: {tx:.3f}m, Y: {ty:.3f}m, Z: {raw_tz:.3f}m"
        f" (xy offsets: {TARGET_FORWARD_OFFSET:.3f}, {TARGET_LATERAL_OFFSET:.3f})"
    )

    workspace_ok, workspace_reason = is_target_in_workspace(tx, ty)
    if not workspace_ok:
        print(f"Target is outside the safe XY workspace: {workspace_reason}. Retrying...")
        return False

    # Use the workspace plane defined by ArUco markers 0/1/2/3 as the pick plane.
    # The target marker only provides XY localization; its apparent height is ignored.
    grasp_z = BOARD_PLANE_PICK_Z
    approach_z = BOARD_PLANE_APPROACH_Z
    print(
        f"Planned grasp Z: {grasp_z:.3f}m, approach Z: {approach_z:.3f}m"
        f" (using board plane from markers 0/1/2/3, raw marker Z ignored: {raw_tz:.3f}m)"
    )

    q_goal = inverse_kinematics(tx, ty, grasp_z)
    q_above = inverse_kinematics(tx, ty, approach_z)
    if q_goal is None or q_above is None:
        print(
            f"IK failed for x={tx:.3f}, y={ty:.3f}, grasp_z={grasp_z:.3f}, approach_z={approach_z:.3f}. Retrying..."
        )
        return False

    print(f"Joint goal: {[round(v, 3) for v in q_goal]}")
    print(f"Joint above: {[round(v, 3) for v in q_above]}")
    
    curr_pos_dict = read_positions(portHandler, packetHandler)
    if curr_pos_dict and all(i in curr_pos_dict for i in [11, 12, 13, 14]):
        q_start = [dynamixel_to_rad(curr_pos_dict[11]),
                   dynamixel_to_rad(curr_pos_dict[12]),
                   dynamixel_to_rad(curr_pos_dict[13]),
                   dynamixel_to_rad(curr_pos_dict[14])]
    else:
        print("Hardware missed a position read (likely motor 13). Retrying calculation...")
        return False
    
    print("Executing Pick Sequence...")
    set_gripper(portHandler, packetHandler, open_gripper=True)
    time.sleep(0.25)
    # Stage 1: rotate only the base joint so the robot faces the target first.
    q_rotate = [q_above[0], q_start[1], q_start[2], q_start[3]]
    print(f"Base-rotate waypoint: {[round(v, 3) for v in q_rotate]}")
    move_smooth(portHandler, packetHandler, q_start, q_rotate, 1.2)
    # Stage 2: move the arm into the pre-grasp pose after the base is aligned.
    move_smooth(portHandler, packetHandler, q_rotate, q_above, 2.0)
    # Stage 3: descend to the pick point.
    move_smooth(portHandler, packetHandler, q_above, q_goal, 1.5)
    set_gripper(portHandler, packetHandler, open_gripper=False)
    time.sleep(0.4)
    move_smooth(portHandler, packetHandler, q_goal, q_above, 2.0)
    
    print("Moving back to default position with payload...")
    q_default = [
        dynamixel_to_rad(DEFAULT_POS[11]),
        dynamixel_to_rad(DEFAULT_POS[12]),
        dynamixel_to_rad(DEFAULT_POS[13]),
        dynamixel_to_rad(DEFAULT_POS[14])
    ]
    move_smooth(portHandler, packetHandler, q_above, q_default, 3.0)
    
    print("Pick sequence completed.")
    return True

def main_sequence():
    pipeline, align, intrinsics = init_realsense()
    detector_params, aruco_dict, detector, OPENCV_OLD = init_aruco()
    portHandler, packetHandler = init_dynamixel()
    
    if not portHandler:
        print("Failed to connect to Dynamixel. Please verify COM port.")
        return

    # Move to default positions immediately
    set_default_positions(portHandler, packetHandler)

    camera_matrix = np.array([
        [intrinsics.fx, 0, intrinsics.ppx],
        [0, intrinsics.fy, intrinsics.ppy],
        [0, 0, 1]
    ], dtype=np.float32)
    dist_coeffs = np.zeros((4,1))
    
    success = False
    try:
        robot_frame = calibrate_workspace(
            pipeline,
            align,
            intrinsics,
            detector_params,
            aruco_dict,
            detector,
            OPENCV_OLD,
        )
        print("System ready. Workspace calibrated. Searching for target object...")
        
        while not success:
            success = execute_task(pipeline, align, intrinsics, detector_params, aruco_dict, detector, OPENCV_OLD, portHandler, packetHandler, camera_matrix, dist_coeffs, robot_frame)
            
    except KeyboardInterrupt:
        print("Manual interrupt received. Shutting down...")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        pipeline.stop()
        cv2.destroyAllWindows()
        print("Closing port while keeping Dynamixel torque PERMANENTLY enabled.")
        portHandler.closePort()


In [81]:
main_sequence()

Moving robot to default position using software interpolation...
Calibrating workspace. Keep markers 0,1,2,3,4 clearly visible...
Calibration progress: 1/8 stable frames
Calibration progress: 2/8 stable frames
Calibration waiting: missing 3D point for marker 3.
Calibration progress: 1/8 stable frames
Calibration progress: 2/8 stable frames
Calibration progress: 3/8 stable frames
Calibration progress: 4/8 stable frames
Calibration progress: 5/8 stable frames
Calibration progress: 6/8 stable frames
Calibration progress: 7/8 stable frames
Calibration progress: 8/8 stable frames
Workspace calibration locked.
Origin: [np.float32(-0.024), np.float32(-0.118), np.float32(0.709)]
X axis: [np.float32(-0.021), np.float32(0.815), np.float32(-0.579)]
Y axis: [np.float32(1.0), np.float32(0.026), np.float32(0.001)]
Z axis: [np.float32(0.016), np.float32(-0.579), np.float32(-0.815)]
System ready. Workspace calibrated. Searching for target object...
Target object detected (ID: 8) at pixel (253, 272), a